In [ ]:
from pathlib import Path
import io
import sys
import subprocess
import time

import requests
import pandas as pd

try:
    from dbnomics import fetch_series
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dbnomics", "-q"])
    from dbnomics import fetch_series

# ==============================================================================
# CONFIG
# ==============================================================================
TARGET_DATASETS = [
    {
        "file_name": "OECD_RD_GDP_2002_2025.csv",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "sdmx_query_key": "all",
        "filters": {"UNIT_MEASURE": "PT_B1GQ", "MEASURE": "G"}
    },
    {
        "file_name": "OECD_Inflation_CPI_2002_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PRICES@DF_PRICES_ALL",
        "sdmx_query_key": ".A.N.CPI.._T.N.GY",
        "filters": {}
    },
    {
        "file_name": "OECD_Unemployment_Rate_2002_2025.csv",
        "agency_id": "OECD.ELS.SAE",
        "dataflow_id": "DSD_LFS@DF_LFS_INDIC",
        "sdmx_query_key": "all",
        "filters": {
            "MEASURE": "UNE_RATE",
            "UNIT_MEASURE": "PT_LF_SUB",
            "SEX": "_T",
            "AGE": "_T"
        }
    },
    {
        "file_name": "OECD_Tertiary_Education_2002_2025.csv",
        "agency_id": "OECD.EDU.IMEP",
        "dataflow_id": "DSD_EAG_LSO_EA@DF_LSO_NEAC_DISTR_EA",
        "sdmx_query_key": "._T.Y25T64.ISCED11A_5T8..........OBS...A",
        "filters": {}
    },
    {
        "file_name": "OECD_GDP_Growth_2002_2025.csv",
        "agency_id": "OECD.SDD.NAD",
        "dataflow_id": "DSD_NAMAIN1@DF_QNA_EXPENDITURE_GROWTH_OECD",
        "sdmx_query_key": "A.Y..S1.S1.B1GQ._Z._Z._Z.PC.L.GY.T0102",
        "filters": {}
    }
]

START_PERIOD = "2002"
END_PERIOD = "2025"
DEFAULT_VERSION = "all"

DATA_RAW_DIR = Path.cwd() / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

PRODUCTIVITY_FILE = "OECD_Productivity_Variation_2002_2025.csv"
PRODUCTIVITY_COUNTRIES = [
    "AUS", "AUT", "BEL", "CAN", "CHE", "CHL", "COL", "CRI", "CZE", "DEU",
    "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "HUN", "IRL", "ISL",
    "ISR", "ITA", "JPN", "KOR", "LTU", "LUX", "LVA", "MEX", "NLD", "NOR",
    "NZL", "POL", "PRT", "SVK", "SVN", "SWE", "TUR", "USA"
]

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "Accept-Encoding": "gzip, deflate",
    "User-Agent": "ResearchScript/2.0"
}

# ==============================================================================
# PART 1 — OECD SDMX REST datasets (generic loop)
# ==============================================================================
for dataset in TARGET_DATASETS:
    print(f"Processing: {dataset['file_name']}...")
    output_path = DATA_RAW_DIR / dataset["file_name"]

    if output_path.exists():
        print("  -> File already exists. Skipping download.\n")
        continue

    query_key = dataset.get("sdmx_query_key", "all")
    dataset_version = dataset.get("version", DEFAULT_VERSION)
    url = (
        f"https://sdmx.oecd.org/public/rest/data/"
        f"{dataset['agency_id']},{dataset['dataflow_id']},{dataset_version}/{query_key}"
    )
    params = {
        "startPeriod": START_PERIOD,
        "endPeriod": END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels"
    }

    response = None
    for attempt in range(4):
        response = requests.get(url, params=params, headers=headers, timeout=90)
        if response.status_code == 403:
            wait = 10 * (attempt + 1)
            print(f"  -> Rate-limited 403. Waiting {wait}s...")
            time.sleep(wait)
            continue
        break

    if response is None:
        print("  -> Request failed completely.\n")
        continue
    if response.status_code != 200:
        print(f"  -> HTTP Error {response.status_code}")
        print("  -> URL:", response.url)
        print("  -> Response preview:", response.text[:500], "\n")
        continue

    df = pd.read_csv(io.StringIO(response.text), low_memory=False)
    if "OBS_VALUE" not in df.columns:
        print("  -> Error: No OBS_VALUE column returned.")
        print("  -> Columns returned:", df.columns.tolist(), "\n")
        continue

    df_clean = df.dropna(subset=["OBS_VALUE"]).copy()

    if "UNIT_MULT" in df_clean.columns:
        df_clean["UNIT_MULT"] = pd.to_numeric(df_clean["UNIT_MULT"], errors="coerce").fillna(0)
        df_clean["OBS_VALUE"] = df_clean["OBS_VALUE"] * (10 ** df_clean["UNIT_MULT"])

    for column_name, filter_value in dataset.get("filters", {}).items():
        if column_name in df_clean.columns:
            before = len(df_clean)
            df_clean = df_clean[df_clean[column_name] == filter_value]
            print(f"  -> Filter {column_name}={filter_value}: {before} -> {len(df_clean)} rows")
        else:
            print(f"  -> Warning: Column '{column_name}' not found.")

    if df_clean.empty:
        print("  -> Error: Dataset is empty after filters.\n")
        continue

    df_tidy = df_clean[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={
        "REF_AREA": "Country", "TIME_PERIOD": "Year", "OBS_VALUE": "Value"
    })
    df_tidy["Year"] = pd.to_numeric(df_tidy["Year"], errors="coerce")
    df_tidy["Value"] = pd.to_numeric(df_tidy["Value"], errors="coerce")
    df_tidy = df_tidy.dropna(subset=["Country", "Year", "Value"])
    df_tidy["Year"] = df_tidy["Year"].astype(int)

    dupes = df_tidy.duplicated(subset=["Country", "Year"], keep=False)
    if dupes.any():
        print(f"  -> WARNING: {dupes.sum()} duplicate country-year rows.")
        df_tidy = df_tidy.groupby(["Country", "Year"], as_index=False)["Value"].mean()

    df_tidy.to_csv(output_path, index=False)
    print(f"  -> Success! Saved {len(df_tidy)} rows.\n")

# ==============================================================================
# PART 2 — Productivity dataset, via DBnomics (different API, same conventions)
# ==============================================================================
print(f"Processing: {PRODUCTIVITY_FILE}...")
productivity_path = DATA_RAW_DIR / PRODUCTIVITY_FILE

if productivity_path.exists():
    print("  -> File already exists. Skipping download.\n")
else:
    series_ids = [
        f"OECD/DSD_PDB@DF_PDB_LV/{c}.A.GDPHRS._T.USD_PPP_H.V._Z._Z._Z"
        for c in PRODUCTIVITY_COUNTRIES
    ]
    df_db = fetch_series(series_ids)
    print("  -> Rows returned:", len(df_db))

    df_prod = df_db.copy()
    df_prod["Country"] = df_prod["series_code"].astype(str).str.split(".").str[0]
    df_prod["Year"] = df_prod["period"].astype(str).str.extract(r"(\d{4})")[0]
    df_prod["Value"] = pd.to_numeric(df_prod["value"], errors="coerce")
    df_prod["Year"] = pd.to_numeric(df_prod["Year"], errors="coerce")
    df_prod = df_prod.dropna(subset=["Country", "Year", "Value"]).copy()
    df_prod["Year"] = df_prod["Year"].astype(int)
    df_prod = df_prod[
        (df_prod["Year"] >= int(START_PERIOD)) & (df_prod["Year"] <= int(END_PERIOD))
    ]
    df_prod = df_prod[["Country", "Year", "Value"]]
    df_prod = df_prod.groupby(["Country", "Year"], as_index=False)["Value"].mean()

    df_prod.to_csv(productivity_path, index=False)
    print(f"  -> Success! Saved {len(df_prod)} rows.\n")

# ==============================================================================
# PART 3 — Verification
# ==============================================================================
print("=" * 60)
print("VERIFICATION")
print("=" * 60)
for file in sorted(DATA_RAW_DIR.glob("*.csv")):
    df = pd.read_csv(file)
    print(file.name, "->", df.shape, "| years:", df["Year"].min(), "-", df["Year"].max())

Processing: OECD_RD_GDP_2000_2025.csv...
  -> File already exists. Skipping download.

Processing: OECD_Inflation_CPI_2000_2025.csv...
  -> File already exists. Skipping download.

Processing: OECD_Unemployment_Rate_2000_2025.csv...
  -> File already exists. Skipping download.

Processing: OECD_Tertiary_Education_2000_2025.csv...
  -> File already exists. Skipping download.

Processing: OECD_GDP_Growth_2000_2025.csv...
  -> File already exists. Skipping download.

Processing: OECD_Productivity_Variation_2000_2025.csv...
  -> Rows returned: 1126
  -> Success! Saved 928 rows.

VERIFICATION
Eurostat_Public_Debt_GDP_2000_2025.csv -> (775, 3) | years: 2000 - 2024
Global_Energy_Dependency_0_100.csv -> (3151, 3) | years: 2000 - 2023
Global_Public_Debt_GDP_2000_2025.csv -> (1856, 3) | years: 2000 - 2024
OECD_GDP_Growth_2000_2025.csv -> (1314, 3) | years: 2000 - 2025
OECD_Inflation_CPI_2000_2025.csv -> (1362, 3) | years: 2000 - 2025
OECD_Productivity_Variation_2000_2025.csv -> (928, 3) | years:

In [3]:
from pathlib import Path
import pandas as pd

DATA_RAW_DIR = Path.cwd() / "Data" / "Raw"

for file in DATA_RAW_DIR.glob("*.csv"):
    df = pd.read_csv(file)
    print(file.name, "->", df.shape, "| years:", df["Year"].min(), "-", df["Year"].max())

Eurostat_Public_Debt_GDP_2000_2025.csv -> (775, 3) | years: 2000 - 2024
Global_Energy_Dependency_0_100.csv -> (3151, 3) | years: 2000 - 2023
Global_Public_Debt_GDP_2000_2025.csv -> (1856, 3) | years: 2000 - 2024
OECD_GDP_Growth_2000_2025.csv -> (1314, 3) | years: 2000 - 2025
OECD_Inflation_CPI_2000_2025.csv -> (1362, 3) | years: 2000 - 2025
OECD_Productivity_Variation_2000_2025.csv -> (928, 3) | years: 2000 - 2024
OECD_RD_GDP_2000_2025.csv -> (1127, 3) | years: 2000 - 2025
OECD_Tertiary_Education_2000_2025.csv -> (1003, 3) | years: 2000 - 2024
OECD_Unemployment_Rate_2000_2025.csv -> (1392, 3) | years: 2000 - 2024
WorldBank_Public_Debt_GDP_2000_2025.csv -> (1131, 3) | years: 2000 - 2024


In [ ]:
import sys
import subprocess
import itertools
from pathlib import Path
import pandas as pd
import requests
import logging

try:
    import country_converter as coco
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "country_converter", "-q"])
    import country_converter as coco

logging.getLogger("country_converter").setLevel(logging.ERROR)

PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

EUROSTAT_BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
WORLD_BANK_BASE = "https://api.worldbank.org/v2"

EUROSTAT_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "DataExtractionScript/4.0"
}

WORLD_BANK_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "DataExtractionScript/4.0"
}

EUROSTAT_GEO_OVERRIDES = {
    "EL": "GR",
    "UK": "GB"
}

def convert_eurostat_geo_to_iso3(df):
    df = df.copy()
    standardized = df["Country"].replace(EUROSTAT_GEO_OVERRIDES)

    df["Country"] = coco.convert(
        names=standardized.tolist(),
        src="ISO2",
        to="ISO3",
        not_found=None
    )

    df = df.dropna(subset=["Country"]).reset_index(drop=True)
    return df


def clean_tidy(df):
    df = df.copy()

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    df = df.dropna(subset=["Country", "Year", "Value"])
    df["Year"] = df["Year"].astype(int)

    df = df[(df["Year"] >= 2002) & (df["Year"] <= 2025)]

    df = (
        df.groupby(["Country", "Year"], as_index=False)["Value"]
        .mean()
        .sort_values(["Country", "Year"])
        .reset_index(drop=True)
    )

    return df


def download_eurostat_public_debt():
    file_name = "Eurostat_Public_Debt_GDP_2002_2025.csv"
    output_path = DATA_RAW_DIR / file_name

    print(f"Processing: {file_name}...")

    if output_path.exists():
        print("  -> File already exists. Skipping.\n")
        return

    params = {
        "format": "JSON",
        "lang": "EN",
        "sector": "S13",
        "unit": "PC_GDP",
        "na_item": "GD",
        "startPeriod": 2002,
        "endPeriod": 2025,
    }

    response = requests.get(
        f"{EUROSTAT_BASE}/gov_10dd_edpt1",
        params=params,
        headers=EUROSTAT_HEADERS,
        timeout=120
    )

    if response.status_code != 200:
        raise RuntimeError(f"Eurostat error {response.status_code}: {response.text[:500]}")

    data = response.json()

    ids = data["id"]
    dimensions = data["dimension"]
    sizes = data["size"]
    values = data["value"]

    ordered_codes = []

    for dim_id in ids:
        categories = dimensions[dim_id]["category"]
        codes = sorted(
            categories["index"],
            key=lambda code: categories["index"][code]
        )
        ordered_codes.append(codes)

    records = []

    for combo in itertools.product(*[range(size) for size in sizes]):
        flat_idx = 0
        stride = 1

        for index in range(len(sizes) - 1, -1, -1):
            flat_idx += combo[index] * stride
            stride *= sizes[index]

        value = values.get(str(flat_idx)) if isinstance(values, dict) else None

        if value is None:
            continue

        row = {
            ids[index]: ordered_codes[index][combo[index]]
            for index in range(len(ids))
        }

        row["OBS_VALUE"] = value
        records.append(row)

    df_raw = pd.DataFrame(records)

    df_tidy = df_raw[["geo", "time", "OBS_VALUE"]].rename(columns={
        "geo": "Country",
        "time": "Year",
        "OBS_VALUE": "Value"
    })

    df_tidy = convert_eurostat_geo_to_iso3(df_tidy)
    df_tidy = clean_tidy(df_tidy)

    df_tidy.to_csv(output_path, index=False)

    print(f"  -> Success! Saved {len(df_tidy)} rows to {output_path}\n")


def get_world_bank_country_codes():
    response = requests.get(
        f"{WORLD_BANK_BASE}/country",
        params={"format": "json", "per_page": 400},
        headers=WORLD_BANK_HEADERS,
        timeout=120
    )

    response.raise_for_status()
    payload = response.json()

    valid_codes = set()

    for item in payload[1]:
        region = item.get("region", {})
        code = item.get("id")

        if region.get("id") != "NA" and code and len(code) == 3:
            valid_codes.add(code)

    return valid_codes


def download_world_bank_dataset(file_name, indicator_code, clip_0_100=False):
    output_path = DATA_RAW_DIR / file_name

    print(f"Processing: {file_name}...")

    if output_path.exists():
        print("  -> File already exists. Skipping.\n")
        return

    response = requests.get(
        f"{WORLD_BANK_BASE}/country/all/indicator/{indicator_code}",
        params={
            "format": "json",
            "per_page": 20000,
            "date": "2002:2025"
        },
        headers=WORLD_BANK_HEADERS,
        timeout=120
    )

    response.raise_for_status()
    payload = response.json()

    valid_country_codes = get_world_bank_country_codes()

    records = []

    for item in payload[1]:
        if not item:
            continue

        country = item.get("countryiso3code")
        year = item.get("date")
        value = item.get("value")

        if not country or country not in valid_country_codes:
            continue

        if value is None:
            continue

        value = float(value)

        if clip_0_100:
            value = max(0.0, min(100.0, value))

        records.append({
            "Country": country,
            "Year": year,
            "Value": value
        })

    df_tidy = pd.DataFrame(records)
    df_tidy = clean_tidy(df_tidy)

    df_tidy.to_csv(output_path, index=False)

    print(f"  -> Success! Saved {len(df_tidy)} rows to {output_path}\n")


def build_global_public_debt():
    print("Processing: Global_Public_Debt_GDP_2002_2025.csv...")

    eurostat_path = DATA_RAW_DIR / "Eurostat_Public_Debt_GDP_2002_2025.csv"
    world_bank_path = DATA_RAW_DIR / "WorldBank_Public_Debt_GDP_2002_2025.csv"
    output_path = DATA_RAW_DIR / "Global_Public_Debt_GDP_2000_2025.csv"

    frames = []

    if eurostat_path.exists():
        eurostat_df = pd.read_csv(eurostat_path)
        frames.append(eurostat_df.assign(_priority=0))

    if world_bank_path.exists():
        world_bank_df = pd.read_csv(world_bank_path)
        frames.append(world_bank_df.assign(_priority=1))

    if not frames:
        raise FileNotFoundError("No public debt source files found.")

    merged = (
        pd.concat(frames, ignore_index=True)
        .sort_values(["Country", "Year", "_priority"])
        .drop_duplicates(["Country", "Year"], keep="first")
        .drop(columns=["_priority"])
        .sort_values(["Country", "Year"])
        .reset_index(drop=True)
    )

    merged = clean_tidy(merged)

    merged.to_csv(output_path, index=False)

    print(f"  -> Success! Saved {len(merged)} rows to {output_path}")
    print("  -> Eurostat preferred where available.\n")


download_eurostat_public_debt()

download_world_bank_dataset(
    file_name="Global_Energy_Dependency_0_100.csv",
    indicator_code="EG.IMP.CONS.ZS",
    clip_0_100=True
)

download_world_bank_dataset(
    file_name="WorldBank_Public_Debt_GDP_2002_2025.csv",
    indicator_code="GC.DOD.TOTL.GD.ZS",
    clip_0_100=False
)

build_global_public_debt()

Processing: Eurostat_Public_Debt_GDP_2000_2025.csv...
  -> File already exists. Skipping.

Processing: Global_Energy_Dependency_0_100.csv...
  -> File already exists. Skipping.

Processing: WorldBank_Public_Debt_GDP_2000_2025.csv...
  -> File already exists. Skipping.

Processing: Global_Public_Debt_GDP_2000_2025.csv...
  -> Success! Saved 1856 rows to c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\Global_Public_Debt_GDP_2000_2025.csv
  -> Eurostat preferred where available.

